## PIDR (personal-information-detection-russian) (осуждаю)

Модель: https://huggingface.co/Ibrahimbaroud/personal-information-detection-russian/tree/main

Датасет: `wolframko/russian-pii-66k` (`source_text` + `privacy_mask`)

Оценка качества модели по целевому BIO-формату из 8 классов:
- `AGE`
- `PERSON`
- `PHONE`
- `EMAIL`
- `ADDRESS`
- `DATE`
- `LOCATION`
- `PID` (класс для документных идентификаторов, включая подтипы вроде паспорт/СНИЛС/ИНН/ОМС/ВУ и др.)

In [1]:
!pip -q install --no-cache-dir \
    "pandas==2.2.2" \
    "pyarrow==18.1.0" \
    "datasets>=2.20.0" \
    "transformers>=4.57.0" \
    "torch" \
    "razdel>=0.5.0" \
    "tqdm" \
    "matplotlib"

In [ ]:
import sys
import itertools
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import pipeline

print("Python:", sys.version)
print("pandas:", pd.__version__)

In [ ]:
MODEL_NAME = "Ibrahimbaroud/personal-information-detection-russian"
DATASET_NAME = "wolframko/russian-pii-66k"

TEXT_COL = "source_text"
RAW_MASK_COL = "privacy_mask"
NERS_COL = "ners"

SAMPLE_SIZE = 600
RANDOM_SEED = 1543
BUFFER_SIZE = 10000
MAX_LENGTH = 512

TARGET_LABELS = [
    "AGE",
    "PERSON",
    "PHONE",
    "EMAIL",
    "ADDRESS",
    "DATE",
    "LOCATION",
    "PID",
]

TAGGING_SCHEME = "BIO"
BIO_LABELS = ["O"] + [f"{prefix}-{label}" for label in TARGET_LABELS for prefix in ["B", "I"]]

TARGET_SCHEMA = {
    "labels": TARGET_LABELS,
    "tagging_scheme": TAGGING_SCHEME,
    "bio_labels": BIO_LABELS,
}

GOLD_LABEL_MAP = {
    "GIVENNAME": "PERSON",
    "SURNAME": "PERSON",
    "TELEPHONENUM": "PHONE",
    "EMAIL": "EMAIL",
    "DATEOFBIRTH": "AGE",
    "STREET": "ADDRESS",
    "BUILDINGNUM": "ADDRESS",
    "CITY": "LOCATION",
    "ZIPCODE": "LOCATION",
    "IDCARDNUM": "PID",
    "DRIVERLICENSENUM": "PID",
    "SOCIALNUM": "PID",
    "TAXNUM": "PID",
    "ACCOUNTNUM": "PID",
    "CREDITCARDNUMBER": "PID",
    "PASSWORD": "PID",
}

MODEL_TAG_TO_TARGET = {
    "PHI-AGE": "AGE",
    "PHI-NAME": "PERSON",
    "PHI-CONTACT_PHONE": "PHONE",
    "PHI-DATE": "DATE",
    "PHI-LOCATION_STREET": "ADDRESS",
    "PHI-LOCATION_HOSPITAL": "ADDRESS",
    "PHI-LOCATION_ZIP": "ADDRESS",
    "PHI-LOCATION_CITY": "LOCATION",
    "PHI-ID": "PID",
}

MODEL_SUPPORTED_TARGETS = sorted(set(MODEL_TAG_TO_TARGET.values()))

print(TARGET_SCHEMA)
print("Supported by model:", MODEL_SUPPORTED_TARGETS)

In [ ]:
ds = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True,
)

ds_shuffled = ds.shuffle(seed=RANDOM_SEED, buffer_size=BUFFER_SIZE)

# выборка из датасета для теста
rows = []
for item in tqdm(
    itertools.islice(ds_shuffled, SAMPLE_SIZE),
    total=SAMPLE_SIZE,
    desc="Sampling",
):
    text = item.get(TEXT_COL) or ""
    privacy_mask = item.get(RAW_MASK_COL) or []

    ners = []
    for ent in privacy_mask:
        if not isinstance(ent, dict):
            continue

        start = ent.get("start")
        end = ent.get("end")
        label = ent.get("label")

        if start is None or end is None or label is None:
            continue

        ners.append({
            "label": str(label).upper(),
            "start_pos": int(start),
            "end_pos": int(end),
        })

    rows.append({TEXT_COL: text, NERS_COL: ners})

df = pd.DataFrame(rows)

print("shape:", df.shape)
print("columns:", list(df.columns))
display(df.head())

In [ ]:
raw_counter = Counter()
mapped_counter = Counter()

for ners in df[NERS_COL]:
    for ent in ners:
        label = str(ent.get("label", "UNKNOWN")).upper()
        raw_counter[label] += 1

        mapped_label = GOLD_LABEL_MAP.get(label)
        if mapped_label is not None:
            mapped_counter[mapped_label] += 1

# частоты меток в датасете
raw_stats = (
    pd.DataFrame(raw_counter.items(), columns=["raw_label", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

mapped_stats = (
    pd.DataFrame(mapped_counter.items(), columns=["mapped_label", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Raw gold labels")
display(raw_stats)

print("Mapped gold labels to 8-class schema")
display(mapped_stats)

In [ ]:
def tokenize_with_model_tokenizer(text):
    """Токенизирует текст tokenizer-ом модели и возвращает токены с char-offsets"""
    encoded = ner_pipe.tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
        add_special_tokens=False,
    )

    tokens = []
    offsets = []

    for start, end in encoded.get("offset_mapping", []):
        start = int(start)
        end = int(end)
        if end <= start:
            continue

        tokens.append(text[start:end])
        offsets.append((start, end))

    return tokens, offsets


def map_gold_entities(ners_list):
    """Нормализует gold-сущности в целевые 8 классов и char-спаны"""
    mapped = []

    for ent in ners_list:
        label = str(ent.get("label", "")).upper()
        mapped_label = GOLD_LABEL_MAP.get(label)
        if mapped_label not in TARGET_LABELS:
            continue

        start = ent.get("start_pos")
        stop = ent.get("end_pos")
        if start is None or stop is None:
            continue

        start = int(start)
        stop = int(stop)
        if stop <= start:
            continue

        mapped.append({"label": mapped_label, "start": start, "end": stop})

    return mapped



def clip_entities_to_char_end(entities, char_end):
    """Обрезает сущности по правой границе видимого окна модели."""
    clipped = []

    for ent in entities:
        start = int(ent["start"])
        end = int(ent["end"])

        if start >= char_end:
            continue

        clipped_end = min(end, char_end)
        if clipped_end <= start:
            continue

        new_ent = dict(ent)
        new_ent["start"] = start
        new_ent["end"] = clipped_end
        clipped.append(new_ent)

    return clipped


def spans_by_label(entities):
    """Группирует список сущностей по целевому label"""
    grouped = defaultdict(list)

    for ent in entities:
        grouped[ent["label"]].append((ent["start"], ent["end"]))

    for label in grouped:
        grouped[label] = sorted(grouped[label], key=lambda x: (x[0], x[1]))

    return grouped


def char_spans_to_token_bio(offsets, spans, label):
    """Переводит char-спаны конкретного класса в BIO по токенам"""
    bio = ["O"] * len(offsets)

    for start, stop in spans:
        touched_ids = []
        for idx, (tok_start, tok_stop) in enumerate(offsets):
            if tok_stop <= start:
                continue
            if tok_start >= stop:
                break
            touched_ids.append(idx)

        if not touched_ids:
            continue

        bio[touched_ids[0]] = f"B-{label}"
        for idx in touched_ids[1:]:
            bio[idx] = f"I-{label}"

    return bio


def merge_class_bio(per_label_bio):
    """Склеивает BIO-последовательности классов в единую BIO-разметку"""
    if not per_label_bio:
        return []

    labels_in_order = TARGET_LABELS
    num_tokens = len(next(iter(per_label_bio.values())))
    merged = ["O"] * num_tokens

    for token_id in range(num_tokens):
        for label in labels_in_order:
            tag = per_label_bio[label][token_id]
            if tag != "O":
                merged[token_id] = tag
                break

    return merged


def tag_type(tag):
    """Возвращает тип сущности из BIO-тега или None для O"""
    if tag == "O":
        return None
    if "-" not in tag:
        return None
    return tag.split("-", 1)[1]

In [ ]:
ner_pipe = pipeline(
    task="token-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    aggregation_strategy="simple",
)  # готовый HF пайплайн для классификации токенов (???)


def get_visible_char_end(text):
    """Возвращает правую char-границу текста, доступного модели"""
    encoded = ner_pipe.tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    visible_char_end = 0
    for start, end in encoded.get("offset_mapping", []):
        if end > visible_char_end:
            visible_char_end = int(end)

    return visible_char_end


def predict_pidr_entities(text):
    """Запускает PIDR и маппит сущности в целевые классы"""
    raw_preds = ner_pipe(text)

    mapped = []

    for ent in raw_preds:
        raw_label = str(ent.get("entity_group", "")).upper()

        start = ent.get("start")
        end = ent.get("end")
        if start is None or end is None:
            continue

        start = int(start)
        end = int(end)
        if end <= start:
            continue

        target_label = MODEL_TAG_TO_TARGET.get(raw_label)
        if target_label is None:
            continue

        mapped.append({
            "label": target_label,
            "start": start,
            "end": end,
            "score": float(ent.get("score", 0.0)),
            "raw_label": raw_label,
        })

    return mapped


def build_row_annotations(text, ners_list):
    """Готовит разметку только на видимой модели части текста"""
    visible_char_end = get_visible_char_end(text)
    visible_text = text[:visible_char_end]

    tokens, offsets = tokenize_with_model_tokenizer(visible_text)

    gold_entities = map_gold_entities(ners_list)
    gold_entities = clip_entities_to_char_end(gold_entities, visible_char_end)

    pred_entities = predict_pidr_entities(visible_text)

    gold_spans = spans_by_label(gold_entities)
    pred_spans = spans_by_label(pred_entities)

    gold_per_label = {}
    pred_per_label = {}

    for label in TARGET_LABELS:
        gold_per_label[label] = char_spans_to_token_bio(offsets, gold_spans.get(label, []), label)
        pred_per_label[label] = char_spans_to_token_bio(offsets, pred_spans.get(label, []), label)

    gold_bio = merge_class_bio(gold_per_label)
    pred_bio = merge_class_bio(pred_per_label)

    return {
        "visible_char_end": visible_char_end,
        "visible_text": visible_text,
        "tokens": tokens,
        "offsets": offsets,
        "gold_entities": gold_entities,
        "pred_entities": pred_entities,
        "gold_bio": gold_bio,
        "pidr_bio": pred_bio,
        "gold_per_label_bio": gold_per_label,
        "pidr_per_label_bio": pred_per_label,
    }

In [ ]:
all_visible_char_end = []
all_visible_text = []
all_is_truncated = []
all_tokens = []
all_offsets = []
all_gold_entities = []
all_pred_entities = []
all_gold_bio = []
all_pred_bio = []
all_gold_per_label_bio = []
all_pred_per_label_bio = []

for row in tqdm(df.itertuples(index=False), total=len(df), desc="PIDR inference"):
    text = getattr(row, TEXT_COL)
    ners_list = getattr(row, NERS_COL)

    ann = build_row_annotations(text, ners_list)

    all_visible_char_end.append(ann["visible_char_end"])
    all_visible_text.append(ann["visible_text"])
    all_is_truncated.append(ann["visible_char_end"] < len(text))
    all_tokens.append(ann["tokens"])
    all_offsets.append(ann["offsets"])
    all_gold_entities.append(ann["gold_entities"])
    all_pred_entities.append(ann["pred_entities"])
    all_gold_bio.append(ann["gold_bio"])
    all_pred_bio.append(ann["pidr_bio"])
    all_gold_per_label_bio.append(ann["gold_per_label_bio"])
    all_pred_per_label_bio.append(ann["pidr_per_label_bio"])

df["visible_char_end"] = all_visible_char_end
df["visible_text"] = all_visible_text
df["is_truncated"] = all_is_truncated
df["tokens"] = all_tokens
df["offsets"] = all_offsets
df["gold_entities_all"] = all_gold_entities
df["pidr_entities_all"] = all_pred_entities
df["gold_bio_all"] = all_gold_bio
df["pidr_bio_all"] = all_pred_bio
df["gold_per_label_bio"] = all_gold_per_label_bio
df["pidr_per_label_bio"] = all_pred_per_label_bio

print("Rows processed:", len(df))
print("Truncated rows:", int(df["is_truncated"].sum()))
display(df[["visible_char_end", "is_truncated", "gold_bio_all", "pidr_bio_all"]].head(3))

## Метрики

Метрики считаются на токеновом уровне отдельно для каждого класса из 8 целевых.

Важно: метрики считаются только на видимой модели части текста (с учетом `truncation=True`, `max_length=MAX_LENGTH`).
Gold-разметка и токены ограничиваются тем же char-окном, что и инференс, а токены для оценки строятся тем же tokenizer-ом, что и у модели.

Класс `PID` считается наравне с остальными и агрегирует документные подтипы (паспорт, СНИЛС, ИНН, ОМС, ВУ и т.д.).

Для выбранного класса (например, `PERSON`):
- `TP` — токен размечен этим классом и в gold, и в prediction;
- `FP` — токен размечен этим классом только в prediction;
- `FN` — токен размечен этим классом только в gold;
- `TN` — токен не принадлежит этому классу ни в gold, ни в prediction.

Далее считаются:
- `precision = TP / (TP + FP)`
- `recall = TP / (TP + FN)`
- `f1 = 2 * precision * recall / (precision + recall)`
- `accuracy = (TP + TN) / (TP + FP + FN + TN)`

Если знаменатель в формуле равен нулю, метрика принимается равной `0.0`.

In [ ]:
def compute_confusion_for_class(df, target_label, gold_col, pred_col):
    """Считает TP, FP, FN, TN для классов"""
    tp = fp = fn = tn = 0

    for gold_tags, pred_tags in zip(df[gold_col], df[pred_col]):
        for g, p in zip(gold_tags, pred_tags):
            gold_pos = tag_type(g) == target_label
            pred_pos = tag_type(p) == target_label

            if gold_pos and pred_pos:
                tp += 1
            elif (not gold_pos) and pred_pos:
                fp += 1
            elif gold_pos and (not pred_pos):
                fn += 1
            else:
                tn += 1

    return tp, fp, fn, tn


def safe_div(numerator, denominator):  # топ5 полезных функций первое место
    """Безопасное деление: возвращает 0.0 при нулевом знаменателе"""
    if denominator == 0:
        return 0.0
    return numerator / denominator


def compute_metrics(tp, fp, fn, tn):
    """Считает precision, recall, f1 и accuracy по confusion counts"""
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)
    accuracy = safe_div(tp + tn, tp + fp + fn + tn)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }


rows = []
for label in TARGET_LABELS:
    tp, fp, fn, tn = compute_confusion_for_class(
        df,
        target_label=label,
        gold_col="gold_bio_all",
        pred_col="pidr_bio_all",
    )
    m = compute_metrics(tp, fp, fn, tn)

    rows.append({
        "label": label,
        "model_support": label in MODEL_SUPPORTED_TARGETS,
        "mapped_pidr_tags": ", ".join(sorted([k for k, v in MODEL_TAG_TO_TARGET.items() if v == label])),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        **m,
    })

metrics_df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)

display(metrics_df)

## Нишевый График

In [ ]:
plot_df = metrics_df[["label", "precision", "recall", "f1", "accuracy"]].copy()

super_colors = [
    "#ff7eb6",  # pink
    "#b388eb",  # lilac
    "#b9f18c",  # light green
    "#7bdff2",  # sky blue
    "#ffd166",  # yellow
    "#f4978e",  # coral
    "#cdb4db",  # pastel violet
    "#90be6d",  # green
]

plt.figure(figsize=(12, 5))
plt.bar(plot_df["label"], plot_df["f1"], color=super_colors[:len(plot_df)])
plt.ylim(0, 1)
plt.ylabel("F1")
plt.title("PIDR: F1 по целевым 8 BIO-классам")
plt.grid(axis="y")

for idx, value in enumerate(plot_df["f1"]):
    plt.text(idx, value + 0.02, f"{value:.3f}", ha="center")

plt.tight_layout()
plt.show()

display(plot_df)

## Как замаплен класс `PID`

Класс `PID` входит в основной набор классов и считается в общей таблице метрик.

На него маппятся:
- предсказания модели с тегом `PHI-ID`;
- gold-подклассы документных/идентификационных сущностей (например, паспорт, СНИЛС, ИНН, ОМС, ВУ и другие).

`USERNAME` в этой версии не маппится и не учитывается в gold-оценке.

In [ ]:
# кто маппится на PID

pid_mapping_rows = []

for raw_label, target_label in sorted(MODEL_TAG_TO_TARGET.items()):
    if target_label == "PID":
        pid_mapping_rows.append({
            "source": "model",
            "raw_label": raw_label,
            "target_label": target_label,
        })

for raw_label, target_label in sorted(GOLD_LABEL_MAP.items()):
    if target_label == "PID":
        pid_mapping_rows.append({
            "source": "gold",
            "raw_label": raw_label,
            "target_label": target_label,
        })

pid_mapping_df = pd.DataFrame(pid_mapping_rows)
display(pid_mapping_df)

## Ошибки:

In [ ]:
error_rows = []

for row_id, row in df.iterrows():
    tokens = list(row["tokens"])
    gold_tags = list(row["gold_bio_all"])
    pred_tags = list(row["pidr_bio_all"])

    for token_id, (token, g, p) in enumerate(zip(tokens, gold_tags, pred_tags)):
        gold_label = tag_type(g)
        pred_label = tag_type(p)

        if gold_label == pred_label:
            continue

        if gold_label is not None and pred_label is None:
            err_type = "FN"
        elif gold_label is None and pred_label is not None:
            err_type = "FP"
        else:
            err_type = "LABEL_SWAP"

        error_rows.append({
            "row_id": row_id,
            "token_id": token_id,
            "token": token,
            "gold": g,
            "pred": p,
            "error_type": err_type,
            "text_preview": row[TEXT_COL][:220],
        })

errors_df = pd.DataFrame(error_rows)

if len(errors_df) == 0:
    print("No token-level errors on sampled data.")
else:
    display(errors_df.head(30))
    display(errors_df["error_type"].value_counts().rename_axis("error_type").reset_index(name="count"))